# rift `rift-biomass-e2e` — visualize DPS crevasse-probability COGs (titiler)

Tile and view the crevasse-model outputs from a **BIOMASS `rift-biomass-e2e`** DPS batch
(hundreds of granules) on a MAAP `titiler` map. Each finished job writes two single-band
float32 probability COGs in `[0, 1]` — **`unet_prob.tif`** (U-Net) and **`gate_prob.tif`**
(RF/CNN gate) — into its DPS output directory. The `biomass-e2e` jobs geocode all four pols of an ESA BIOMASS L1A SCS granule, derive `*_intensity.tif` COGs, then run the nisar-crevasse BIOMASS gate + 4-channel U-Net model, writing `gate_prob.tif` + `unet_prob.tif` per granule.

This notebook makes **two separate maps** — one U-Net mosaic and one gate mosaic — each a
single **MosaicJSON** layer combining every granule in the run, served by MAAP's shared
titiler (`https://titiler.maap-project.org`, the exact endpoint the MAAP visualization
tutorials use for COGs).

**How it finds the outputs.** After a DPS run, MAAP stages each job's `output/` to a deeply
nested key
`~/my-private-bucket/dps_output/<algo>/<version>/<tag>/YYYY/MM/DD/HH/MM/SS/<hash>/`.
Rather than guess those paths, we read the **submission CSV** the driver notebook wrote to
`~/my-public-bucket/dps_submission_results/`, take each `job_id`, and ask MAAP
`get_job_result(job_id)` for the exact output directory — so every layer is tied back to its
granule.

**Private-bucket caveat.** The DPS outputs live in `my-private-bucket`
(`s3://maap-ops-workspace/<user>/…`). Every MAAP titiler *tutorial* tiles from the **shared**
bucket, so we **try the private `s3://` URL first** and, only if titiler can't read it
(HTTP≠200), **stage** the COGs into `~/my-public-bucket` (= `s3://maap-ops-workspace/shared/<user>/…`,
which titiler can read) and tile those.

Run this in a **MAAP Hub workspace** (maap-py v5 available).

**Prereqs**
- A completed `rift-biomass-e2e` DPS batch and its submission CSV (from the `biomass_e2e_dps_runner` notebook).
- `pip install cogeo-mosaic ipyleaflet httpx` (titiler/maap-py are already on the Hub).

In [ ]:
# One-time in a fresh workspace:
# %pip install cogeo-mosaic ipyleaflet httpx
import os, glob, json, re, shutil
import pandas as pd
import httpx
from ipyleaflet import Map, TileLayer, LayersControl, ScaleControl, FullScreenControl, basemaps
from cogeo_mosaic.mosaic import MosaicJSON
from maap.maap import MAAP

maap = MAAP()
TITILER = "https://titiler.maap-project.org"   # MAAP shared COG/MosaicJSON tiler
username = maap.profile.account_info()["username"]

PRIVATE_LOCAL = os.path.expanduser("~/my-private-bucket/")
PUBLIC_LOCAL  = os.path.expanduser("~/my-public-bucket/")
PRIVATE_S3    = f"s3://maap-ops-workspace/{username}/"
PUBLIC_S3     = f"s3://maap-ops-workspace/shared/{username}/"
print("user:", username)
print("titiler:", TITILER)

## 1. Locate the submission CSV for the run

The `rift-biomass-e2e` driver saved one CSV per submission to
`~/my-public-bucket/dps_submission_results/biomass-e2e_*.csv`, with a `job_id` column and a
`item_id` column naming each granule. We pick the **newest** matching CSV by
default — set `CSV_PATH` explicitly to visualize a specific earlier run.

**No CSV?** §1b falls back to `maap.list_jobs()` to recover the finished job_ids for this algorithm directly from MAAP.

In [ ]:
SUB_DIR = os.path.join(PUBLIC_LOCAL, "dps_submission_results")
pattern = os.path.join(SUB_DIR, "biomass-e2e_*.csv")
csvs = sorted(glob.glob(pattern))
print(f"{len(csvs)} submission CSV(s) matching biomass-e2e_*.csv:")
for c in csvs:
    print("  ", os.path.basename(c))

# Newest by filename (the driver stamps them ..._YYYYMMDDHHMM.csv). Override to pick a run.
CSV_PATH = csvs[-1] if csvs else None

jobs = []          # list of (job_id, granule_label)
if CSV_PATH:
    sub = pd.read_csv(CSV_PATH)
    LABEL_COL = "item_id" if "item_id" in sub.columns else sub.columns[1]
    jobs = [(str(r["job_id"]).strip(), str(r.get(LABEL_COL, "")).strip())
            for _, r in sub.iterrows()
            if pd.notna(r.get("job_id")) and str(r["job_id"]).strip()]
    print(f"\nusing {os.path.basename(CSV_PATH)} -> {len(jobs)} job_ids (label col: {LABEL_COL})")
    display(sub.head())
else:
    print(f"\nNo submission CSV under {SUB_DIR} matching biomass-e2e_*.csv "
          "-- the next cell recovers job_ids from maap.list_jobs().")

## 1b. Fallback: recover job_ids from MAAP if there's no submission CSV

If the batch was submitted without saving a CSV (or you're on a fresh workspace), ask MAAP for
the finished jobs of this algorithm directly. `list_jobs(get_job_details=False)` returns a
compact `{id, tags}` list; we keep the completed ones and label each by its submit **tag**
(the per-granule name isn't in the compact listing — set `RUN_TAG_FILTER` to narrow to one
batch, or leave it blank for all completed jobs of `rift-biomass-e2e`).

This cell only does anything when the CSV lookup above found nothing — set
`FORCE_LIST_JOBS = True` to use it regardless. Adjust `VERSION` / `STATUS` / `RUN_TAG_FILTER`
as needed; depending on your maap-py build you may need to pass the numeric `process_id`
instead of the string `algo_id` (see §4 of the driver notebook for the numeric id).

In [ ]:
ALGO_ID         = "rift-biomass-e2e"    # OGC process string id for this pipeline
VERSION         = ""                  # e.g. "main"; "" = any version
STATUS          = "job-completed"     # only finished jobs have staged outputs
RUN_TAG_FILTER  = ""                  # e.g. "biomass-e2e-thwaites"; "" = all tags
FORCE_LIST_JOBS = False               # True = use list_jobs even if the CSV loaded jobs

if not jobs or FORCE_LIST_JOBS:
    kw = {"algo_id": ALGO_ID, "status": STATUS, "get_job_details": False}
    if VERSION:        kw["version"] = VERSION
    if RUN_TAG_FILTER: kw["tag"] = RUN_TAG_FILTER
    print(f"recovering job_ids: maap.list_jobs({kw})")
    r = maap.list_jobs(**kw)
    if hasattr(r, "json"):                       # v5 Response, or v4 requests.Response
        try:
            body = r.json()
        except Exception:
            body = json.loads(getattr(r, "text", "{}") or "{}")
    else:
        body = r
    jlist = body.get("jobs", []) if isinstance(body, dict) else (body or [])
    recovered = []
    for j in jlist:
        jid  = j.get("id") or j.get("job_id") or j.get("payload_id")
        if not jid:
            continue
        tags = j.get("tags") or []
        recovered.append((str(jid), tags[0] if tags else str(jid)))
    jobs = recovered
    print(f"recovered {len(jobs)} completed job(s) for {ALGO_ID}"
          + (f" tag={RUN_TAG_FILTER}" if RUN_TAG_FILTER else ""))
    for jid, lab in jobs[:5]:
        print("  ", jid, "|", lab)

assert jobs, ("No job_ids from the CSV or list_jobs -- check ALGO_ID/VERSION/STATUS/"
              "RUN_TAG_FILTER above, or confirm the batch has finished.")

## 2. Resolve each job's output directory and find its prob COGs

`get_job_result(job_id)` returns the job's staged output location. Its exact shape varies
across maap-py builds (a list of hrefs, a JSON body, or a WPS-XML string), so we pull the
first `s3://…/dps_output/…` URL out of whatever it returns and re-root it to a canonical
`s3://maap-ops-workspace/<user>/dps_output/…` directory and its local mount. Then we glob that
directory for `unet_prob.tif` / `gate_prob.tif`. Jobs that aren't finished (no result yet) are
skipped and counted.

In [ ]:
def _result_text(job_id):
    """Return a string blob from get_job_result across maap-py shapes."""
    r = maap.get_job_result(job_id)
    if hasattr(r, "status_code"):                 # v5 OGC: Response object
        if r.status_code != 200:
            return ""
        try:
            body = r.json()
            return body if isinstance(body, str) else json.dumps(body)
        except Exception:
            return r.text or ""
    try:                                          # v4: indexable list of hrefs
        return "\n".join(str(x) for x in r)
    except TypeError:
        return str(r)

def output_dir_s3(job_id):
    """First s3://.../dps_output/... URL from the job result, canonicalized."""
    text = _result_text(job_id)
    cands = [u.rstrip("/") for u in re.findall(r"s3://[^\s<>\"']+", text) if "dps_output" in u]
    if not cands:
        return None
    i = cands[0].find("/dps_output/")
    return f"s3://maap-ops-workspace/{username}/{cands[0][i+1:]}"   # canonical <user>/dps_output/...

def to_local(s3url):
    return s3url.replace(PRIVATE_S3, PRIVATE_LOCAL)

def find_cogs(job_id):
    """-> dict(unet_s3, gate_s3, unet_local, gate_local, dir_s3, dir_local) or None.

    ``dir_s3``/``dir_local`` are the job's output directory, so later cells can glob it
    for the per-pol intensity COGs without re-calling get_job_result."""
    d_s3 = output_dir_s3(job_id)
    if not d_s3:
        return None
    d_local = to_local(d_s3)
    out = {}
    for prod, name in (("unet", "unet_prob.tif"), ("gate", "gate_prob.tif")):
        hits = glob.glob(os.path.join(d_local, "**", name), recursive=True)
        if hits:
            local = hits[0]
            rel = os.path.relpath(local, d_local).replace(os.sep, "/")
            out[f"{prod}_local"] = local
            out[f"{prod}_s3"] = f"{d_s3}/{rel}"
    if not out:
        return None
    out["dir_s3"] = d_s3
    out["dir_local"] = d_local
    return out

records = []
missing = 0
for job_id, label in jobs:
    cogs = find_cogs(job_id)
    if not cogs:
        missing += 1
        continue
    records.append({"job_id": job_id, "label": label, **cogs})

found = pd.DataFrame(records)
print(f"{len(found)} jobs with COGs on disk; {missing} skipped (unfinished / no output yet)")
if len(found):
    print(f"  U-Net COGs: {found['unet_s3'].notna().sum()}   Gate COGs: {found['gate_s3'].notna().sum()}")
    print("  example:", found.iloc[0].get('unet_s3'))
found.head()

## 3. Make the COGs tileable (private `s3://` first, stage to public on 403)

We probe titiler once with the first private COG. If titiler can read it, we tile the private
URLs directly (no copying). If not, we copy each COG into `~/my-public-bucket/crevasse_viz/…`
(namespaced per run + product + job so the identically-named `unet_prob.tif` files don't
collide) and tile the resulting shared-bucket `s3://` URLs.

In [ ]:
RUN_TAG = os.path.splitext(os.path.basename(CSV_PATH))[0]   # namespace staged copies per run

def titiler_can_read(s3url):
    try:
        return httpx.get(f"{TITILER}/cog/info", params={"url": s3url}, timeout=30).status_code == 200
    except Exception:
        return False

def stage_to_public(local_path, prod, job_id):
    subdir = os.path.join(PUBLIC_LOCAL, "crevasse_viz", RUN_TAG, prod, job_id)
    os.makedirs(subdir, exist_ok=True)
    dst = os.path.join(subdir, os.path.basename(local_path))
    if not os.path.exists(dst):
        shutil.copy2(local_path, dst)
    return dst.replace(PUBLIC_LOCAL, PUBLIC_S3)

# Probe once on the first available U-Net (or gate) COG.
probe = next((r["unet_s3"] for r in records if r.get("unet_s3")),
             next((r["gate_s3"] for r in records if r.get("gate_s3")), None))
PRIVATE_OK = bool(probe) and titiler_can_read(probe)
print("titiler can read the private bucket directly:", PRIVATE_OK)
if not PRIVATE_OK:
    print("-> staging COGs into ~/my-public-bucket/crevasse_viz/ for tiling (this copies files)")

def tileable(local_path, s3url, prod, job_id):
    if PRIVATE_OK:
        return s3url
    return stage_to_public(local_path, prod, job_id)

unet_urls, gate_urls = [], []
for r in records:
    if r.get("unet_s3"):
        unet_urls.append(tileable(r["unet_local"], r["unet_s3"], "unet", r["job_id"]))
    if r.get("gate_s3"):
        gate_urls.append(tileable(r["gate_local"], r["gate_s3"], "gate", r["job_id"]))
print(f"{len(unet_urls)} U-Net URLs, {len(gate_urls)} gate URLs ready to tile")

## 4. Mosaic + render helpers

`MosaicJSON.from_urls` reads each COG's header to build the tile index, we POST it to
`{TITILER}/mosaics`, and read back the mosaic's `tilejson.json`. Render params for a `[0,1]`
probability raster: `rescale="0,1"`, `colormap_name`, `nodata="nan"`, `bidx=1`. The prob COGs
are on the Antarctic grid (EPSG:3031); titiler reprojects to WebMercator for the slippy map.

In [ ]:
def gamma_formula(gamma=0.5, band="R"):
    """titiler ``color_formula`` string for a numpy-style gamma of ``out = in**gamma``.

    titiler applies ``color_formula`` *after* the linear p2-p98 ``rescale`` casts the band
    to uint8 and *before* the colormap, so gamma acts on well-defined [0,255] data (baking a
    256-entry gamma LUT into ``colormap=`` instead blows the tile-request URL past titiler's
    length limit -> the request errors and the tilejson comes back with no ``bounds``).
    titiler's gamma op is ``in**(1/g)`` (g>1 brightens), so a numpy ``in**0.5`` brighten is
    ``g = 1/0.5 = 2``. gamma<1 lifts the mid/low tones so the very-bright / very-dark pol
    imagery shows detail instead of clipping to black & white. ``band='R'`` = band 1 (these
    are single-band rasters)."""
    return f"gamma {band} {1.0 / gamma}"

def make_mosaic_tilejson(urls, colormap_name=None, color_formula=None, rescale="0,1"):
    """Build a MosaicJSON from COG URLs, register it, and return its rendered tilejson dict.

    ``colormap_name`` is a titiler-named cmap ('magma', 'gray', ...); ``color_formula`` is an
    optional color-ops string (e.g. from ``gamma_formula``) applied after ``rescale``."""
    mosaic = MosaicJSON.from_urls(urls)
    reg = httpx.post(
        f"{TITILER}/mosaics",
        headers={"Content-Type": "application/vnd.titiler.mosaicjson+json"},
        json=mosaic.model_dump(exclude_none=True),
        timeout=180,
    ).json()
    tj_href = next(l["href"] for l in reg["links"] if l.get("rel") == "tilejson")
    params = {"rescale": rescale, "bidx": 1, "nodata": "nan"}
    if colormap_name is not None:
        params["colormap_name"] = colormap_name
    if color_formula is not None:
        params["color_formula"] = color_formula
    return httpx.get(tj_href, params=params, timeout=60).json()

def show_mosaic(urls, title, colormap_name=None, color_formula=None, rescale="0,1"):
    if not urls:
        print(f"No COGs for {title}."); return None
    tj = make_mosaic_tilejson(urls, colormap_name=colormap_name, color_formula=color_formula, rescale=rescale)
    if "bounds" not in tj:   # titiler returned an error payload instead of a tilejson
        print(f"titiler did not return a tilejson for {title}: {tj}"); return None
    b = tj["bounds"]
    m = Map(center=((b[1] + b[3]) / 2, (b[0] + b[2]) / 2),
            zoom=(tj.get("minzoom", 3) or 3) + 1,
            basemap=basemaps.Esri.WorldImagery, scroll_wheel_zoom=True)
    m.add_layer(TileLayer(url=tj["tiles"][0], name=title, opacity=0.9))
    m.add_control(LayersControl(position="topright"))
    m.add_control(ScaleControl(position="bottomleft"))
    m.add_control(FullScreenControl(position="topleft"))   # corner button -> browser fullscreen
    extra = f" | color_formula={color_formula}" if color_formula else ""
    print(f"{title}: {len(urls)} granules | cmap={colormap_name} | rescale={rescale}{extra} | bounds={[round(x,3) for x in b]}")
    return m

def pct_stretch(s3url, lo=2, hi=98):
    """Ask titiler for a COG's p{lo}-p{hi} band stats -> 'min,max' rescale string.

    Intensity COGs (amp**2) are not in [0,1] like the probability rasters, so a fixed
    rescale renders them nearly black; this derives a per-COG stretch. Falls back to
    "0,1" if statistics are unavailable."""
    try:
        r = httpx.get(f"{TITILER}/cog/statistics",
                      params={"url": s3url, "p": [lo, hi]}, timeout=60)
        b1 = next(iter(r.json().values()))   # first (only) band
        return f"{b1[f'percentile_{lo}']},{b1[f'percentile_{hi}']}"
    except Exception as e:
        print(f"  stretch probe failed ({e}); falling back to 0,1")
        return "0,1"

## 5. U-Net crevasse-probability mosaic (BIOMASS)

All granules' `unet_prob.tif` combined into one MosaicJSON layer. `magma` on `[0,1]`: dark =
low probability, bright = likely crevasse.

In [ ]:
show_mosaic(unet_urls, "U-Net crevasse prob", colormap_name="magma")

## 6. Gate crevasse-probability mosaic (BIOMASS)

All granules' `gate_prob.tif` combined into one MosaicJSON layer (`viridis` on `[0,1]`) — the
gate stage's per-tile probability, separate from the U-Net above.

In [ ]:
show_mosaic(gate_urls, "Gate crevasse prob", colormap_name="viridis")

## 7. Per-polarization intensity mosaics (BIOMASS)

The four **intensity** COGs (`*_<POL>_intensity.tif`, = amplitude²) the crevasse BIOMASS model
takes as its 4-channel input, one mosaic per polarization (HH, HV, VH, VV). Intensity has an
open-ended range (not `[0,1]` like the probability rasters), so each pol gets its own **p2–p98
percentile stretch** (`pct_stretch`) to drop the extreme bright/dark outliers, then a **0.5
gamma** (`gamma_formula`, sent as titiler's `color_formula`) that lifts the mid/low tones so
features read instead of crushing to black & white. The cell below gathers the URLs per pol;
the four cells after it render one map each.

In [ ]:
# Collect the per-pol intensity COGs (*_<POL>_intensity.tif) across every granule.
POLS = ["HH", "HV", "VH", "VV"]
intensity_urls = {p: [] for p in POLS}
for r in records:
    d_local, d_s3 = r["dir_local"], r["dir_s3"]
    for pol in POLS:
        for local in glob.glob(os.path.join(d_local, "**", f"*_{pol}_intensity.tif"), recursive=True):
            rel = os.path.relpath(local, d_local).replace(os.sep, "/")
            intensity_urls[pol].append(tileable(local, f"{d_s3}/{rel}", f"{pol}_intensity", r["job_id"]))

print("intensity COGs per pol:", {p: len(v) for p, v in intensity_urls.items()})

In [ ]:
POL = "HH"
if intensity_urls.get(POL):
    stretch = pct_stretch(intensity_urls[POL][0])   # per-pol p2-p98 (intensity isn't [0,1])
    display(show_mosaic(intensity_urls[POL], f"{POL} intensity", colormap_name="gray",
                        color_formula=gamma_formula(0.5), rescale=stretch))  # p2-p98 clip + 0.5 gamma
else:
    print(f"No *_{POL}_intensity.tif found.")

In [ ]:
POL = "HV"
if intensity_urls.get(POL):
    stretch = pct_stretch(intensity_urls[POL][0])   # per-pol p2-p98 (intensity isn't [0,1])
    display(show_mosaic(intensity_urls[POL], f"{POL} intensity", colormap_name="gray",
                        color_formula=gamma_formula(0.5), rescale=stretch))  # p2-p98 clip + 0.5 gamma
else:
    print(f"No *_{POL}_intensity.tif found.")

In [ ]:
POL = "VH"
if intensity_urls.get(POL):
    stretch = pct_stretch(intensity_urls[POL][0])   # per-pol p2-p98 (intensity isn't [0,1])
    display(show_mosaic(intensity_urls[POL], f"{POL} intensity", colormap_name="gray",
                        color_formula=gamma_formula(0.5), rescale=stretch))  # p2-p98 clip + 0.5 gamma
else:
    print(f"No *_{POL}_intensity.tif found.")

In [ ]:
POL = "VV"
if intensity_urls.get(POL):
    stretch = pct_stretch(intensity_urls[POL][0])   # per-pol p2-p98 (intensity isn't [0,1])
    display(show_mosaic(intensity_urls[POL], f"{POL} intensity", colormap_name="gray",
                        color_formula=gamma_formula(0.5), rescale=stretch))  # p2-p98 clip + 0.5 gamma
else:
    print(f"No *_{POL}_intensity.tif found.")

## 8. (optional) Inspect a single granule

Drill into one granule's U-Net COG at full resolution (single-COG `/cog` tiler, not the
mosaic). Set `IDX` to a row in the `found` table.

In [ ]:
IDX = 0
if len(found):
    row = found.iloc[IDX]
    url = tileable(row["unet_local"], row["unet_s3"], "unet", row["job_id"]) if row.get("unet_s3") else None
    if url:
        tj = httpx.get(f"{TITILER}/cog/tilejson.json",
                       params={"url": url, "rescale": "0,1", "bidx": 1,
                               "colormap_name": "magma", "nodata": "nan"}, timeout=60).json()
        b = tj["bounds"]
        m = Map(center=((b[1]+b[3])/2, (b[0]+b[2])/2), zoom=(tj.get("minzoom", 5) or 5)+2,
                basemap=basemaps.Esri.WorldImagery, scroll_wheel_zoom=True)
        m.add_layer(TileLayer(url=tj["tiles"][0], name=str(row["label"]), opacity=0.9))
        m.add_control(LayersControl(position="topright"))
        print("granule:", row["label"], "| job:", row["job_id"])
        display(m)
    else:
        print("Row", IDX, "has no U-Net COG.")
else:
    print("No granules resolved — run the cells above.")